# Congklak AlphaZero Training (Kaggle Version)
### Metode: gdown (Initial Download) + WandB (Automatic Auto-Save)

**🚀 PANDUAN RINGKAS KAGGLE:**
1. **Internet & GPU:** Pastikan Settings (kanan) -> Internet **ON** & Accelerator **GPU**.
2. **Secrets:** Menu Add-ons -> Secrets -> `WANDB_API_KEY` harus **Attached**.
3. **Persistence:** Kaggle itu sementara. Simpan hasil permanen di **WandB**.
4. **Training Background:** Klik 'Save Version' -> 'Save & Run All' agar training jalan terus meski laptop mati.

**🔄 ALUR FILE:**
GitHub (Kode) ➔ GDrive (Model Awal) ➔ Kaggle (Training) ➔ WandB (Backup Model)

In [ ]:
# 1. Install dependencies
!pip install --upgrade gdown wandb --quiet

import os, sys, torch, wandb, warnings
from kaggle_secrets import UserSecretsClient
warnings.filterwarnings('ignore', category=SyntaxWarning)

# 2. Setup WandB (Ambil API Key dari Kaggle Secrets)
user_secrets = UserSecretsClient()
try:
    wandb_key = user_secrets.get_secret("WANDB_API_KEY")
    wandb.login(key=wandb_key)
    print("WandB logged in successfully!")
except:
    print("PERINGATAN: WANDB_API_KEY tidak ditemukan di Kaggle Secrets. Silahkan tambahkan di menu Add-ons.")

repo_url = "https://github.com/billdansr/AlphaDDA.git"
local_repo_path = "/kaggle/working/AlphaDDA"
game_subdir = "AlphaZero/Congklak"

# 3. Clone Repository
if not os.path.exists(local_repo_path):
    print("Cloning repository...")
    !git clone {repo_url} {local_repo_path}
else:
    print("Cleaning and updating existing repository...")
    # Menghindari error merge conflict jika ada perubahan lokal
    !git -C {local_repo_path} reset --hard origin/main && git -C {local_repo_path} pull

local_game_path = os.path.join(local_repo_path, game_subdir)
os.chdir(local_game_path)

# 4. Download Model Terbaru dari GDrive (Opsional: Jika ingin melanjutkan training)
# Pastikan file di GDrive sudah di-set 'Anyone with the link can view'
GDRIVE_FILE_ID = '1F7ssgFZpEYJu8MhN3a0RNjk6d6o-EyBM'

print("--- VERIFIKASI CHECKPOINT ---")
# Hapus file lama jika ada yang rusak atau ingin benar-benar sinkron ulang
if os.path.exists('checkpoint.model'):
    !rm checkpoint.model

if GDRIVE_FILE_ID != 'MASUKKAN_FILE_ID_DISINI':
    print(f"📡 Mengunduh model terbaru dari GDrive...")
    !gdown {GDRIVE_FILE_ID} -O checkpoint.model

if os.path.exists('checkpoint.model'):
    try:
        # Memastikan model terbaca sebelum training berat dimulai
        ckpt = torch.load('checkpoint.model', map_location='cpu', weights_only=True)
        it = ckpt.get('iteration', 0)
        print(f"✅ Model Terdeteksi: Melanjutkan dari Iterasi {it}")
        # Copy juga ke nama spesifik jika train_mp.py mencarinya
        !cp checkpoint.model checkpoint_{it}.model
    except Exception as e:
        print(f"❌ File rusak atau tidak valid: {e}")
else:
    print("⚠️ Tidak ada model ditemukan. Training akan dimulai dari nol.")

# Pastikan permission file benar untuk eksekusi script
!chmod +x *.model 2>/dev/null || true

In [ ]:
# 5. Jalankan Training dengan Auto-Sync (Mencegah Data Hilang saat Timeout)
import threading, time
%env PYTHONPATH=.:$PYTHONPATH

# Inisialisasi WandB untuk tracking eksperimen skripsi
run = wandb.init(
    project="AlphaZero-Congklak", 
    name=f"train-{os.getenv('KAGGLE_KERNEL_RUN_TYPE', 'interactive')}-{time.strftime('%H%M')}"
)

# Fungsi background untuk upload model ke WandB setiap 30 menit
def auto_sync():
    print("📡 Background Auto-Sync aktif. Progres akan di-backup setiap 30 menit.")
    while True:
        try:
            time.sleep(1800) # Tunggu 30 menit
            if os.path.exists('checkpoint.model'):
                wandb.save("checkpoint.model")
                print(f"✅ Auto-sync berhasil pada {time.strftime('%H:%M:%S')}")
        except Exception as e:
            print(f"⚠️ Auto-sync tertunda: {e}")

# Jalankan tracker di thread terpisah
threading.Thread(target=auto_sync, daemon=True).start()

!python train_mp.py

# 6. Simpan Model ke WandB secara otomatis
print("\n--- UPLOADING RESULTS TO WANDB ---")
wandb.save("*.model")
wandb.save("training_log.csv")
run.finish()
print("Selesai! Model dan Log aman di cloud WandB.")

In [ ]:
# 7. Evaluasi (Opsional)
%cd /kaggle/working/AlphaDDA/AlphaDDA1/Congklak
# Ambil model hasil training tadi
!cp /kaggle/working/AlphaDDA/AlphaZero/Congklak/checkpoint.model .
!python test_dda.py 1 300